In [1]:
# Auto reload notebook:
%load_ext autoreload
%autoreload 2

In [4]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    force=True  
)
import sys
sys.path.append("../")

from models.bot_detector import BotDetector

/Users/trokhymovych/Documents/GitHub/AI-powered-bot-detection-API/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Testing logic module: 

In [6]:
bot_detector = BotDetector(model_name="../data/mbert_trained")
texts = [
    "I love this product! It's amazing and works perfectly.",
    "This is the worst experience I've ever had. Totally disappointed.",
    "Not bad, but could be better. It does the job.",
    "Absolutely fantastic! Exceeded my expectations.",
    "Terrible service. Will never buy from here again.",
    "It's okay, not great but not terrible either.",
    "I'm very happy with my purchase. Highly recommend it!",
    "Worst product ever. It broke after one use.",
    "Decent quality for the price. Satisfied with it.",
    "I would give it 5 stars! Excellent value and performance."
    "This product is a scam. Don't waste your money."
]
results = bot_detector.predict(texts)
results

2026-02-18 21:11:12,536 | models.bot_detector | INFO | Loading model: ../data/mbert_trained
/Users/trokhymovych/Documents/GitHub/AI-powered-bot-detection-API/venv/lib/python3.9/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


{'is_bot': True,
 'confidence': 0.6279,
 'text_scores': [0.1351,
  0.9048,
  0.3579,
  0.9968,
  0.3636,
  0.9408,
  0.8521,
  0.0198,
  0.9986,
  0.71],
 'num_texts': 10}

# Testing API endpoints:

In [20]:
import pandas as pd
import numpy as np
fox8 = pd.read_csv("../data/fox8_scores.csv")
fox8.head()

# Select sample of users: 
unique_users = fox8["user_id"].unique()
sample_users = np.random.choice(unique_users, size=300, replace=False)
fox8_sample = fox8[fox8["user_id"].isin(sample_users)]
fox8_sample.head()

,text,label,scores,scores_ms,scores_osmdet,scores_with_context,scores_no_context,user_id
397,RT @DPZaragoza: Los #bomberos de la @DPZaragoz...,0,0.977959,0.004139,0.154994,0.993411,0.999139,297051227
398,RT @jmmulet: A todos los entusiastas de la lec...,0,0.016923,0.200239,0.001124,0.172930,0.000540,297051227
399,@KaminariFace @KaminariFace Debo der de la der...,0,0.006744,0.005494,0.001992,0.013293,0.000087,297051227
400,@crpandemonium Googlear el nombre del nuevo ve...,0,0.024606,0.562056,0.001158,0.228388,0.000116,297051227
401,Ojalá los manifiestos de mañana fueran la mita...,0,0.006953,0.022720,0.035212,0.025510,0.000655,297051227


In [21]:
import requests

def api_predict(texts):
        url = "http://localhost:8000/predict"
        payload = {
        "texts": texts
        }

        response = requests.post(url, json=payload)
        result = response.json()
        return result

In [23]:
from tqdm import tqdm
fox8_input = fox8_sample.groupby(["user_id", "label"]).text.agg(list).to_dict()
prediction_results = {}
for key, texts in tqdm(fox8_input.items()):
    result = api_predict(texts[:20])  # Limit to first 20 texts for API input
    prediction_results[key] = result

100%|██████████| 300/300 [05:38<00:00,  1.13s/it]


In [26]:
prediction_df = pd.DataFrame(prediction_results.values())
prediction_df["true"] = [key[1] for key in prediction_results.keys()]
prediction_df

,is_bot,confidence,text_scores,num_texts,detail,true
0,False,0.2320,"[0.0125, 0.0854, 0.9716, 0.0057, 0.0158, 0.002...",20.0,NaN,0
1,False,0.2179,"[0.0093, 0.0032, 0.0871, 0.9978, 0.9574, 0.861...",20.0,NaN,0
2,True,0.5806,"[0.0609, 0.9994, 0.002, 0.1102, 0.9856, 0.0712...",20.0,NaN,0
3,False,0.2820,"[0.019, 0.37, 0.0056, 0.9936, 0.0245, 0.0502, ...",20.0,NaN,0
4,False,0.3623,"[0.052, 0.907, 0.0224, 0.3162, 0.7785, 0.9962,...",20.0,NaN,0
...,...,...,...,...,...,...
295,True,0.7126,"[0.7103, 0.6632, 0.9837, 0.5645, 0.041, 0.9577...",20.0,NaN,1
296,True,0.5098,"[0.1735, 0.5153, 0.3721, 0.7258, 0.487, 0.9991...",20.0,NaN,1
297,True,0.6490,"[0.9991, 0.9885, 0.5645, 0.142, 0.7982, 0.3486...",20.0,NaN,1
298,True,0.6391,"[0.978, 0.0913, 0.5134, 0.9648, 0.0893, 0.895,...",20.0,NaN,1


In [37]:
from sklearn.metrics import roc_auc_score, classification_report
prediction_df.dropna(subset=["confidence"], inplace=True)  # Drop rows where confidence is NaN

auc = roc_auc_score(prediction_df.true, prediction_df.confidence.fillna(0))
print("fox8_23 ROC AUC:", auc)
print(classification_report(prediction_df.true, prediction_df.is_bot.astype(int)))

fox8_23 ROC AUC: 0.9437086092715231
              precision    recall  f1-score   support

           0       0.91      0.90      0.90       151
           1       0.90      0.90      0.90       147

    accuracy                           0.90       298
   macro avg       0.90      0.90      0.90       298
weighted avg       0.90      0.90      0.90       298

